In [ ]:
!cp "/content/drive/MyDrive/Hate Speech LLMs/ARR Data/LLM Evaluation/deepSeekV3Data.xlsx" .

In [ ]:
import numpy as np
import pandas as pd
import ast
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score

In [ ]:
df = pd.read_excel("deepSeekV3Data.xlsx")

In [ ]:
ref_col = 'Hop_3_Tactic (Ref)'
pred_col = 'Hop_3_Tactic'

In [ ]:
df = df[df["Hop_1_Triage (Ref)"] != "بی‌ضرر"]
df[pred_col] = df[pred_col].fillna('[]')

In [ ]:
refTactics = []
for refTactic in df[ref_col]:
  refTactics.extend(ast.literal_eval(refTactic))
set(refTactics)

{'انسان\u200cزدایی',
 'تحریک به خشونت/آسیب',
 'ترویج کلیشه\u200cهای مضر',
 'دشنام/زبان تحقیرآمیز',
 'فراخوان برای تبعیض/حذف',
 'نظریه\u200cهای توطئه'}

In [ ]:
llmTactics = []
for llmTactic in df[pred_col]:
  llmTactics.extend(ast.literal_eval(llmTactic))
set(llmTactics)

{'انحطاط',
 'انسان\u200cزدایی',
 'تحریک به خشونت/آسیب',
 'ترویج کلیشه\u200cهای مضر',
 'خیانت',
 'دشمن/زبان تحقیرآمیز',
 'دشنام/زبان تحقیرآمیز',
 'زبان تحقیرآمیز',
 'فراخوان برای تبعیض/حذف',
 'نظریه\u200cهای توطئه'}

In [ ]:
df[ref_col] = df[ref_col].apply(ast.literal_eval)
df[pred_col] = df[pred_col].apply(ast.literal_eval)

In [ ]:
mlb = MultiLabelBinarizer()

mlb.fit(list(df[ref_col]) + list(df[pred_col]))

y_true = mlb.transform(df[ref_col])
y_pred = mlb.transform(df[pred_col])

In [ ]:
def f1_adapted(y_true_bin, y_pred_bin):
    """
    Adapted F1 from arXiv:2506.19073 using binarized arrays.
    A prediction is correct if ANY label overlaps for a sample.
    """

    hits = []

    for yt, yp in zip(y_true_bin, y_pred_bin):
        # skip empty ground truth (recommended)
        if not yt.any():
            continue

        hit = int((yt & yp).any())
        hits.append(hit)

    return np.mean(hits) if hits else 0.0

In [ ]:
f1_micro = f1_score(y_true, y_pred, average='micro')
f1_macro = f1_score(y_true, y_pred, average='macro')
f1_adapted = f1_adapted(y_true, y_pred)

print(f"Micro F1: {f1_micro:.4f}")
print(f"Macro F1: {f1_macro:.4f}")
print(f"Adapted F1: {f1_adapted:.4f}")

Micro F1: 0.7581
Macro F1: 0.3593
Adapted F1: 0.9188
